# Supervised Learning EDA: Single-Word Dataset Preparation

**Goal**: Create a preliminary one-word dataset for the supervised learning component.

**Tasks**:
1. Filter clues to single-word definitions AND single-word answers
2. Explore data quality issues
3. Familiarize with WordNet for synonym prediction
4. Assess dataset size for Coach review

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Use relative path (works when notebook is opened from Google Drive)
DATA_DIR = "../data"
NOTEBOOK_DIR = "."

print(f"Current working directory: {os.getcwd()}")
print(f"Data directory: {os.path.abspath(DATA_DIR)}")

# Verify the path exists
if os.path.exists(DATA_DIR):
    print(f"\n✓ Data directory found!")
    print(f"Files available: {os.listdir(DATA_DIR)}")
else:
    print(f"\n✗ Data directory not found!")
    print("Make sure you opened this notebook directly from Google Drive.")

In [ ]:
# Install required packages
try:
    import nltk
    from nltk.corpus import wordnet as wn
    wn.synsets("test")
except:
    import nltk
    nltk.download('wordnet', quiet=True)
    nltk.download('omw-1.4', quiet=True)
    from nltk.corpus import wordnet as wn

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# For Google Colab - uncomment these lines
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT = "/content/drive/MyDrive/Milestone II - NLP Cryptic Crossword Clues"

# For local runs
PROJECT_ROOT = os.path.abspath("..")
DATA_DIR = f"{PROJECT_ROOT}/data"

print(f"Data directory: {DATA_DIR}")
print(f"Files available: {os.listdir(DATA_DIR)}")

In [ ]:
# Load the main clues dataset
df_clues = pd.read_csv(f'{DATA_DIR}/clues_raw.csv')
print(f"Total clues loaded: {len(df_clues):,}")
df_clues.head()

## 2. Data Quality Assessment

In [ ]:
# Check for missing values in key columns
print("=== Missing Values ===")
missing = df_clues[['clue', 'answer', 'definition']].isnull().sum()
print(missing)
print(f"\nPercentage missing:")
print((missing / len(df_clues) * 100).round(2))

In [ ]:
# Check data types
print("=== Data Types ===")
print(df_clues[['clue', 'answer', 'definition']].dtypes)

# Check for non-string values in answer/definition columns
print(f"\n=== Non-string values ===")
print(f"Answers that are not strings: {df_clues['answer'].apply(lambda x: not isinstance(x, str)).sum()}")
print(f"Definitions that are not strings: {df_clues['definition'].apply(lambda x: not isinstance(x, str)).sum()}")

In [ ]:
# Look at some problematic rows
print("=== Sample rows with missing definitions ===")
df_clues[df_clues['definition'].isnull()].head(10)

In [ ]:
# Check for square brackets (known parsing issues)
has_brackets = df_clues['clue'].str.contains(r'\[', na=False)
print(f"Clues with square brackets (likely mis-parsed): {has_brackets.sum():,}")
print(f"Percentage: {has_brackets.sum() / len(df_clues) * 100:.2f}%")

# Show examples
print("\n=== Examples of clues with brackets ===")
df_clues[has_brackets][['clue', 'answer', 'definition']].head(5)

## 3. Creating the Single-Word Dataset

In [ ]:
# Start with a clean copy, dropping rows with missing answer or definition
df_clean = df_clues.dropna(subset=['answer', 'definition']).copy()
print(f"After dropping NaN: {len(df_clean):,} clues ({len(df_clean)/len(df_clues)*100:.1f}%)")

In [ ]:
# Remove clues with square brackets (mis-parsed)
df_clean = df_clean[~df_clean['clue'].str.contains(r'\[', na=False)]
print(f"After removing bracketed clues: {len(df_clean):,} clues")

In [ ]:
# Calculate word counts for answer and definition
# Handle potential issues with non-string values
df_clean['answer_wc'] = df_clean['answer'].astype(str).str.split().str.len()
df_clean['definition_wc'] = df_clean['definition'].astype(str).str.split().str.len()

print("=== Answer Word Count Distribution ===")
print(df_clean['answer_wc'].value_counts().head(10))

print("\n=== Definition Word Count Distribution ===")
print(df_clean['definition_wc'].value_counts().head(10))

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Answer word count
ax1 = axes[0]
df_clean['answer_wc'].value_counts().sort_index().head(10).plot(kind='bar', ax=ax1)
ax1.set_title('Answer Word Count Distribution')
ax1.set_xlabel('Number of Words')
ax1.set_ylabel('Count')

# Definition word count
ax2 = axes[1]
df_clean['definition_wc'].value_counts().sort_index().head(15).plot(kind='bar', ax=ax2)
ax2.set_title('Definition Word Count Distribution')
ax2.set_xlabel('Number of Words')
ax2.set_ylabel('Count')

plt.tight_layout()
plt.savefig(f'{DATA_DIR}/../notebooks/word_count_distributions.png', dpi=150)
plt.show()

In [ ]:
# Create the single-word dataset: BOTH answer AND definition must be single words
df_one_word = df_clean[
    (df_clean['answer_wc'] == 1) &
    (df_clean['definition_wc'] == 1)
].copy()

print(f"=== SINGLE-WORD DATASET ===")
print(f"Total clues with single-word answer AND definition: {len(df_one_word):,}")
print(f"Percentage of original dataset: {len(df_one_word)/len(df_clues)*100:.2f}%")

In [ ]:
# Also check: single-word answers only (more relaxed)
df_one_word_answer = df_clean[df_clean['answer_wc'] == 1].copy()
print(f"Clues with single-word ANSWERS (any definition): {len(df_one_word_answer):,}")

# Single-word definitions only
df_one_word_def = df_clean[df_clean['definition_wc'] == 1].copy()
print(f"Clues with single-word DEFINITIONS (any answer): {len(df_one_word_def):,}")

In [ ]:
# Clean up the single-word dataset: lowercase and strip whitespace
df_one_word['answer_clean'] = df_one_word['answer'].str.lower().str.strip()
df_one_word['definition_clean'] = df_one_word['definition'].str.lower().str.strip()

# Preview the dataset
print("=== Sample of Single-Word Dataset ===")
df_one_word[['clue', 'definition_clean', 'answer_clean']].sample(10)

## 4. Dataset Statistics for Coach Review

In [ ]:
# Key statistics
print("=" * 60)
print("DATASET SIZE SUMMARY FOR COACH REVIEW")
print("=" * 60)
print(f"Original dataset:                     {len(df_clues):>10,} clues")
print(f"After removing NaN:                   {len(df_clean):>10,} clues")
print(f"Single-word answer & definition:      {len(df_one_word):>10,} clues")
print(f"Unique definitions:                   {df_one_word['definition_clean'].nunique():>10,}")
print(f"Unique answers:                       {df_one_word['answer_clean'].nunique():>10,}")
print(f"Unique (definition, answer) pairs:    {len(df_one_word.groupby(['definition_clean', 'answer_clean'])):>10,}")
print("=" * 60)

In [ ]:
# Most common definitions
print("=== Top 20 Most Common Definitions ===")
print(df_one_word['definition_clean'].value_counts().head(20))

In [ ]:
# Most common answers
print("=== Top 20 Most Common Answers ===")
print(df_one_word['answer_clean'].value_counts().head(20))

In [ ]:
# How many times does each (definition, answer) pair appear?
pair_counts = df_one_word.groupby(['definition_clean', 'answer_clean']).size().reset_index(name='count')
print("=== Distribution of (Definition, Answer) Pair Frequencies ===")
print(pair_counts['count'].describe())
print(f"\nPairs appearing only once: {(pair_counts['count'] == 1).sum():,}")
print(f"Pairs appearing 2+ times: {(pair_counts['count'] >= 2).sum():,}")
print(f"Pairs appearing 5+ times: {(pair_counts['count'] >= 5).sum():,}")

## 5. WordNet Exploration for Supervised Learning

In [ ]:
from nltk.corpus import wordnet as wn

# Example: Look up synsets for a definition word
example_def = "anger"
synsets = wn.synsets(example_def)

print(f"=== WordNet synsets for '{example_def}' ===")
for syn in synsets:
    print(f"\n{syn.name()}: {syn.definition()}")
    print(f"  Lemmas: {[l.name() for l in syn.lemmas()]}")

In [ ]:
# Function to get all synonyms for a word from WordNet
def get_wordnet_synonyms(word):
    """Get all lemma names (synonyms) for a word across all synsets."""
    synonyms = set()
    for syn in wn.synsets(word):
        for lemma in syn.lemmas():
            # Clean up the lemma name (replace underscores with spaces)
            synonyms.add(lemma.name().lower().replace('_', ' '))
    # Remove the word itself
    synonyms.discard(word.lower())
    return synonyms

# Test it
print(f"Synonyms for 'anger': {get_wordnet_synonyms('anger')}")
print(f"Synonyms for 'happy': {get_wordnet_synonyms('happy')}")

In [ ]:
# Check: How often is the answer a direct WordNet synonym of the definition?
def answer_in_synonyms(row):
    """Check if the answer is in the WordNet synonyms of the definition."""
    definition = row['definition_clean']
    answer = row['answer_clean']
    synonyms = get_wordnet_synonyms(definition)
    return answer in synonyms

# Sample to test (full dataset would take a while)
sample_size = 5000
df_sample = df_one_word.sample(min(sample_size, len(df_one_word)), random_state=42).copy()

print(f"Testing on {len(df_sample):,} samples...")
df_sample['answer_is_synonym'] = df_sample.apply(answer_in_synonyms, axis=1)

synonym_rate = df_sample['answer_is_synonym'].mean()
print(f"\n=== WordNet Synonym Match Rate ===")
print(f"Answer is a direct WordNet synonym of definition: {synonym_rate*100:.2f}%")
print(f"({df_sample['answer_is_synonym'].sum()} out of {len(df_sample)} samples)")

In [ ]:
# Examples where answer IS a synonym
print("=== Examples: Answer IS a WordNet synonym ===")
matches = df_sample[df_sample['answer_is_synonym']][['definition_clean', 'answer_clean', 'clue']].head(10)
for _, row in matches.iterrows():
    print(f"  {row['definition_clean']} -> {row['answer_clean']}")

In [ ]:
# Examples where answer is NOT a synonym (these are the challenging cases)
print("=== Examples: Answer is NOT a direct WordNet synonym ===")
non_matches = df_sample[~df_sample['answer_is_synonym']][['definition_clean', 'answer_clean', 'clue']].head(15)
for _, row in non_matches.iterrows():
    print(f"  {row['definition_clean']} -> {row['answer_clean']}")
    print(f"    Clue: {row['clue'][:80]}...")

In [ ]:
# WordNet path similarity between definition and answer
from nltk.corpus import wordnet as wn

def max_path_similarity(word1, word2):
    """Get maximum path similarity between any synsets of two words."""
    synsets1 = wn.synsets(word1)
    synsets2 = wn.synsets(word2)

    if not synsets1 or not synsets2:
        return None

    max_sim = 0
    for s1 in synsets1:
        for s2 in synsets2:
            sim = s1.path_similarity(s2)
            if sim and sim > max_sim:
                max_sim = sim
    return max_sim if max_sim > 0 else None

# Test
print(f"Path similarity 'anger' to 'rage': {max_path_similarity('anger', 'rage')}")
print(f"Path similarity 'anger' to 'happy': {max_path_similarity('anger', 'happy')}")
print(f"Path similarity 'dog' to 'cat': {max_path_similarity('dog', 'cat')}")

In [ ]:
# Calculate path similarity for our sample
print("Calculating path similarities (this may take a minute)...")
df_sample['path_similarity'] = df_sample.apply(
    lambda row: max_path_similarity(row['definition_clean'], row['answer_clean']),
    axis=1
)

# Statistics
print("\n=== Path Similarity Statistics ===")
print(df_sample['path_similarity'].describe())

# How many have no path (different parts of speech or not in WordNet)
no_path = df_sample['path_similarity'].isna().sum()
print(f"\nPairs with no WordNet path: {no_path} ({no_path/len(df_sample)*100:.1f}%)")

In [ ]:
# Visualize path similarity distribution
plt.figure(figsize=(10, 5))
df_sample['path_similarity'].dropna().hist(bins=30)
plt.title('Distribution of WordNet Path Similarity (Definition -> Answer)')
plt.xlabel('Path Similarity')
plt.ylabel('Count')
plt.axvline(x=1.0, color='r', linestyle='--', label='Identical/Direct synonym')
plt.legend()
plt.savefig(f'{DATA_DIR}/../notebooks/path_similarity_distribution.png', dpi=150)
plt.show()

## 6. Data Quality Issues Summary

In [ ]:
# Check for potential issues in our single-word dataset

# 1. Non-alphabetic characters in definitions/answers
has_nonalpha_def = df_one_word['definition_clean'].str.contains(r'[^a-z]', na=False)
has_nonalpha_ans = df_one_word['answer_clean'].str.contains(r'[^a-z]', na=False)

print("=== Data Quality Issues ===")
print(f"Definitions with non-alpha characters: {has_nonalpha_def.sum():,}")
print(f"Answers with non-alpha characters: {has_nonalpha_ans.sum():,}")

# Show examples
if has_nonalpha_def.sum() > 0:
    print("\nExamples of definitions with non-alpha:")
    print(df_one_word[has_nonalpha_def]['definition_clean'].head(10).tolist())

In [ ]:
# 2. Very short definitions (1-2 characters) - likely parsing errors
short_defs = df_one_word[df_one_word['definition_clean'].str.len() <= 2]
print(f"\nDefinitions with 1-2 characters: {len(short_defs):,}")
if len(short_defs) > 0:
    print("Examples:")
    print(short_defs[['definition_clean', 'answer_clean', 'clue']].head(10))

In [ ]:
# 3. Check if definition equals answer (trivial cases)
trivial = df_one_word[df_one_word['definition_clean'] == df_one_word['answer_clean']]
print(f"\nCases where definition == answer: {len(trivial):,}")
if len(trivial) > 0:
    print("Examples:")
    print(trivial[['definition_clean', 'answer_clean', 'clue']].head(10))

In [ ]:
# Create a "clean" version removing obvious issues
df_one_word_clean = df_one_word[
    ~has_nonalpha_def &
    ~has_nonalpha_ans &
    (df_one_word['definition_clean'].str.len() > 2) &
    (df_one_word['definition_clean'] != df_one_word['answer_clean'])
].copy()

print(f"\n=== FINAL CLEAN SINGLE-WORD DATASET ===")
print(f"Size: {len(df_one_word_clean):,} clues")
print(f"Unique definitions: {df_one_word_clean['definition_clean'].nunique():,}")
print(f"Unique answers: {df_one_word_clean['answer_clean'].nunique():,}")

## 7. Save the Single-Word Dataset

In [ ]:
# Save the clean single-word dataset for supervised learning
output_cols = ['clue_id', 'clue', 'answer', 'definition', 'answer_clean', 'definition_clean', 'source']
df_one_word_clean[output_cols].to_csv(f'{DATA_DIR}/clues_single_word.csv', index=False)
print(f"Saved: {DATA_DIR}/clues_single_word.csv")
print(f"Shape: {df_one_word_clean[output_cols].shape}")

## 8. Summary for Coach Meeting

In [ ]:
print("="*70)
print("SUMMARY FOR COACH MEETING")
print("="*70)
print("\n1. DATASET SIZE:")
print(f"   - Original clues: {len(df_clues):,}")
print(f"   - Single-word (definition & answer): {len(df_one_word):,}")
print(f"   - After cleaning: {len(df_one_word_clean):,}")
print(f"   - This is {len(df_one_word_clean)/len(df_clues)*100:.1f}% of the original data")

print("\n2. DATA QUALITY ISSUES FOUND:")
print(f"   - Missing definitions: {df_clues['definition'].isna().sum():,}")
print(f"   - Bracketed clues (mis-parsed): {has_brackets.sum():,}")
print(f"   - Non-alphabetic definitions: {has_nonalpha_def.sum():,}")
print(f"   - Very short definitions (<=2 chars): {len(short_defs):,}")
print(f"   - Trivial (definition == answer): {len(trivial):,}")

print("\n3. WORDNET ANALYSIS (on 5000 sample):")
print(f"   - Direct synonym matches: {synonym_rate*100:.1f}%")
print(f"   - Mean path similarity: {df_sample['path_similarity'].mean():.3f}")
print(f"   - No WordNet path found: {no_path/len(df_sample)*100:.1f}%")

print("\n4. QUESTIONS FOR COACH:")
print(f"   - Is {len(df_one_word_clean):,} clues sufficient for supervised learning?")
print("   - Should we include multi-word definitions to increase data size?")
print("   - How to handle definition/answer pairs not in WordNet?")
print("="*70)